# HW1 — MDPs & Dynamic Programming
### Bellman Equations · Policy Evaluation · Policy Improvement
**Yogeshvar Reddy Kallam** · IST 597 Deep RL · Penn State Spring 2025

---

## Problem 1: On Thin Ice — FrozenLake MDP Analysis

**Grid:** `H  F  F  S₀  F  F  G`

**Policy:** π(left|s) = 0.5, π(right|s) = 0.5  
**Goal:** Derive the value function analytically using the Bellman equation, analyse return variance, and apply greedy policy improvement.

In [ ]:
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt


### Part (a) — Closed-Form Bellman Solution

In [ ]:
def compute_value_closed_form(Pg, Ph, gamma):
    """V^π(S₀) = (Pg - Ph) / (1 - Ps*γ)"""
    Ps = 1.0 - Pg - Ph
    return (Pg - Ph) / (1 - Ps * gamma)

# Symmetric grid: Pg ≈ Ph = 1/6
Pg, Ph, gamma = 1/6, 1/6, 0.9
V = compute_value_closed_form(Pg, Ph, gamma)
print(f"V^π(S₀) = {V:.4f}")

Var_G0 = Pg*(1-V)**2 + Ph*(-1-V)**2
print(f"Var(G₀) = {Var_G0:.4f}")

N = int(np.ceil(Var_G0 / 0.1**2))
print(f"Min episodes for ε=0.1 (CLT): N ≈ {N:,}")


### Part (b) — Monte Carlo Policy Evaluation

In [ ]:
env = gym.make("FrozenLake-v1", is_slippery=True)
n_s, n_a = env.observation_space.n, env.action_space.n
gamma = 0.9

V_mc = np.zeros(n_s)
counts = np.zeros(n_s)

for _ in range(50_000):
    s, _ = env.reset()
    traj = []
    done = False
    while not done:
        a = env.action_space.sample()
        ns, r, done, _, _ = env.step(a)
        traj.append((s, r))
        s = ns
    G = 0.0
    for st, rw in reversed(traj):
        G = rw + gamma * G
        V_mc[st] += G
        counts[st] += 1

V_mc = np.where(counts > 0, V_mc / counts, 0.0)
print("Value function (4x4 grid):")
print(V_mc.reshape(4,4).round(3))
env.close()


### Part (c) — Greedy Policy Improvement

In [ ]:
env = gym.make("FrozenLake-v1", is_slippery=True)
n_s, n_a = env.observation_space.n, env.action_space.n

def policy_eval(env, pi, gamma=0.9, theta=1e-4):
    V = np.zeros(n_s)
    while True:
        delta = 0
        for s in range(n_s):
            v = sum(pi[s,a] * sum(p*(r + gamma*(0 if d else V[ns]))
                    for p,ns,r,d in env.unwrapped.P[s][a])
                    for a in range(n_a))
            delta = max(delta, abs(v - V[s]))
            V[s] = v
        if delta < theta: break
    return V

pi = np.ones((n_s, n_a)) / n_a
V  = policy_eval(env, pi)

# Greedy improvement
pi_star = np.zeros((n_s, n_a))
for s in range(n_s):
    Q = [sum(p*(r + 0.9*(0 if d else V[ns]))
             for p,ns,r,d in env.unwrapped.P[s][a]) for a in range(n_a)]
    pi_star[s, np.argmax(Q)] = 1.0

names = ["←","↓","→","↑"]
print("Optimal actions:")
print(np.array([names[np.argmax(pi_star[s])] for s in range(n_s)]).reshape(4,4))
env.close()
